In [62]:
import pandas as pd

In [63]:
data = pd.read_csv('processed_data.csv')

In [64]:
data.head()

,Home ID,Appliance Type,Energy Consumption (kWh),Outdoor Temperature (°C),Season,Household Size,timestamp
0,94,Fridge,0.20,-1.0,Fall,2,2023-12-02 21:12:00
1,435,Oven,0.23,31.1,Summer,5,2023-08-06 20:11:00
2,466,Dishwasher,0.32,21.3,Fall,3,2023-11-21 06:39:00
3,496,Heater,3.92,-4.2,Winter,1,2023-01-21 21:56:00
4,137,Microwave,0.44,34.5,Summer,5,2023-08-26 04:31:00


In [65]:
datacopy = data
datacopy['timestamp'] = pd.to_datetime(datacopy['timestamp'])
datacopy.head()
# df_energy['hour'] = df_energy['timestamp'].dt.hour

# print(df_energy[['hour']].head())

,Home ID,Appliance Type,Energy Consumption (kWh),Outdoor Temperature (°C),Season,Household Size,timestamp
0,94,Fridge,0.20,-1.0,Fall,2,2023-12-02 21:12:00
1,435,Oven,0.23,31.1,Summer,5,2023-08-06 20:11:00
2,466,Dishwasher,0.32,21.3,Fall,3,2023-11-21 06:39:00
3,496,Heater,3.92,-4.2,Winter,1,2023-01-21 21:56:00
4,137,Microwave,0.44,34.5,Summer,5,2023-08-26 04:31:00


In [66]:
# datacopy['hour'] = datacopy['timestamp'].dt.hour
type(datacopy['timestamp'])

pandas.core.series.Series

In [67]:
datacopy['timestamp'].dtype

dtype('<M8[ns]')

In [68]:
datacopy['hour'] = datacopy['timestamp'].dt.hour
datacopy['day'] = datacopy['timestamp'].dt.day
datacopy['month'] = datacopy['timestamp'].dt.month
datacopy['week'] = datacopy['timestamp'].dt.weekday

In [69]:
datacopy = datacopy.sort_values(by=['Appliance Type', 'timestamp'])

In [70]:
datacopy.head()

,Home ID,Appliance Type,Energy Consumption (kWh),Outdoor Temperature (°C),Season,Household Size,timestamp,hour,day,month,week
24828,9,Air Conditioning,4.42,23.0,Winter,1,2023-01-01 00:42:00,0,1,1,6
94809,409,Air Conditioning,4.37,36.0,Winter,1,2023-01-01 01:32:00,1,1,1,6
17701,475,Air Conditioning,3.24,13.0,Winter,5,2023-01-01 01:46:00,1,1,1,6
58518,179,Air Conditioning,3.75,15.3,Winter,1,2023-01-01 02:09:00,2,1,1,6
24785,480,Air Conditioning,4.22,8.6,Winter,5,2023-01-01 02:31:00,2,1,1,6


In [71]:
datacopy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99049 entries, 24828 to 18655
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Home ID                   99049 non-null  int64         
 1   Appliance Type            99049 non-null  object        
 2   Energy Consumption (kWh)  99049 non-null  float64       
 3   Outdoor Temperature (°C)  99049 non-null  float64       
 4   Season                    99049 non-null  object        
 5   Household Size            99049 non-null  int64         
 6   timestamp                 99049 non-null  datetime64[ns]
 7   hour                      99049 non-null  int32         
 8   day                       99049 non-null  int32         
 9   month                     99049 non-null  int32         
 10  week                      99049 non-null  int32         
dtypes: datetime64[ns](1), float64(2), int32(4), int64(2), object(2)
memory usage: 7.6

In [72]:
datacopy['lag_1'] = datacopy['Energy Consumption (kWh)'].shift(1)
datacopy['lag_24'] = datacopy['Energy Consumption (kWh)'].shift(24)

In [73]:
datacopy['rolling_mean_24'] = (
    datacopy['Energy Consumption (kWh)']
    .rolling(window=24)
    .mean()
)

In [74]:
datacopy.shape

(99049, 14)

In [75]:
datacopy.head()

,Home ID,Appliance Type,Energy Consumption (kWh),Outdoor Temperature (°C),Season,Household Size,timestamp,hour,day,month,week,lag_1,lag_24,rolling_mean_24
24828,9,Air Conditioning,4.42,23.0,Winter,1,2023-01-01 00:42:00,0,1,1,6,NaN,NaN,NaN
94809,409,Air Conditioning,4.37,36.0,Winter,1,2023-01-01 01:32:00,1,1,1,6,4.42,NaN,NaN
17701,475,Air Conditioning,3.24,13.0,Winter,5,2023-01-01 01:46:00,1,1,1,6,4.37,NaN,NaN
58518,179,Air Conditioning,3.75,15.3,Winter,1,2023-01-01 02:09:00,2,1,1,6,3.24,NaN,NaN
24785,480,Air Conditioning,4.22,8.6,Winter,5,2023-01-01 02:31:00,2,1,1,6,3.75,NaN,NaN


In [76]:
datacopy.isnull().sum()

Home ID                      0
Appliance Type               0
Energy Consumption (kWh)     0
Outdoor Temperature (°C)     0
Season                       0
Household Size               0
timestamp                    0
hour                         0
day                          0
month                        0
week                         0
lag_1                        1
lag_24                      24
rolling_mean_24             23
dtype: int64

In [77]:
datacopy = datacopy.dropna()

In [78]:
datacopy.shape

(99025, 14)

In [79]:
appliances = datacopy['Appliance Type'].unique()
print(f'Type of Appliances: {len(appliances)} ')
appliances


Type of Appliances: 10 


array(['Air Conditioning', 'Computer', 'Dishwasher', 'Fridge', 'Heater',
       'Lights', 'Microwave', 'Oven', 'TV', 'Washing Machine'],
      dtype=object)

In [80]:
test = datacopy[datacopy['Appliance Type'] == 'Computer'].copy()

In [86]:
for  appliance in appliances:
    device = datacopy[datacopy['Appliance Type'] == appliance].copy()
    x = device.drop(['Energy Consumption (kWh)','Appliance Type', 'Season','Home ID','timestamp'], axis = 1)
    y = device['Energy Consumption (kWh)']
    split_index = int(len(device) * 0.85)

    x_train = x.iloc[: split_index]
    y_train = y.iloc[: split_index]

    x_test = x.iloc[split_index :] 
    y_test = y.iloc[split_index :]

    x_train.to_csv(f'Processed_data_for_each_device/x_train_{appliance}.csv', index=False)
    y_train.to_csv(f'Processed_data_for_each_device/y_train_{appliance}.csv', index=False)
    x_test.to_csv(f'Processed_data_for_each_device/x_test_{appliance}.csv', index=False)
    y_test.to_csv(f'Processed_data_for_each_device/y_test_{appliance}.csv', index=False)
    



In [87]:
x_train.head()

,Outdoor Temperature (°C),Household Size,hour,day,month,week,lag_1,lag_24,rolling_mean_24
92541,2.7,3,0,1,1,6,0.95,0.90,1.150417
38068,5.0,2,1,1,1,6,1.12,1.51,1.170000
91598,-1.6,4,2,1,1,6,1.98,0.21,1.230833
51529,38.9,4,2,1,1,6,1.67,0.63,1.243333
21658,12.6,3,5,1,1,6,0.93,1.11,1.240000


In [88]:
y_train.head()

92541    1.12
38068    1.98
91598    1.67
51529    0.93
21658    1.03
Name: Energy Consumption (kWh), dtype: float64

In [89]:
x_test.head()

,Outdoor Temperature (°C),Household Size,hour,day,month,week,lag_1,lag_24,rolling_mean_24
59815,14.5,5,18,4,11,5,0.76,1.95,1.366250
47134,21.6,3,19,4,11,5,1.80,1.94,1.299167
68612,-8.0,3,19,4,11,5,0.33,1.98,1.270833
55434,0.7,2,21,4,11,5,1.30,1.91,1.259583
57979,14.0,5,21,4,11,5,1.64,0.91,1.262500


In [90]:
y_test.head()

59815    1.80
47134    0.33
68612    1.30
55434    1.64
57979    0.98
Name: Energy Consumption (kWh), dtype: float64